# Solar Filament Segmentation Challenge 2026
Author: shepsci

This notebook loads the exact weights in **our repository**. The owned alternative mirror is `shepscientific/filament-segmentation-2026-review-assets`. No third-party model handle is downloaded. Run `git lfs pull` when cloning. Preserve upstream attribution in NOTICE and weights/manifest.json.

Default execution performs final inference only. Optional diagnostics and new scratch training are disabled. The included weights are reused public pretrained assets. Their complete earlier training chain was not independently reconstructed.

## 1. Install the recorded environment
Use Python 3.12.12 and the package versions below. The exact replay used Apple MPS. CUDA and CPU can differ at numerical thresholds. Consult README for resource requirements before execution.

In [ ]:
%pip install -r requirements.txt

## 2. Configure official input paths and check our local weights
Set paths to the original competition distribution. Paths identify files only and are not prediction features.

In [ ]:
from pathlib import Path
import json
import inference
IMAGES = Path("/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images")
TRAIN = Path("/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/train")
OUTPUT = Path("work")
assert IMAGES.is_dir(), "Set IMAGES to official test/test_images"
weights = inference.restore_weights()
print("Owned local weights:", weights)
print(json.loads(Path("architecture.json").read_text())[0]["head_class"])

## 3. Run the complete final method
Read the supplied grayscale pixels through the pinned predictor. Run both models at 2048, confidence 0.5, box-IoU argument 0, and maximum 100 detections. Keep available predictions from either model. Suppress pooled masks above IoU 0.5. Assign each pixel to at most one mask in confidence order. Remove remnants below five pixels and write compressed COCO RLE.

inference.py contains the reusable prediction and output modules. No labels, metadata features, neighbor images, or reference predictions enter this step.

In [ ]:
receipt = inference.main(IMAGES, OUTPUT / "submission.csv")
receipt

## 4. Validate and compare with the recorded submission
The full reference run processed 180 images and emitted 1,317 masks in 178 images. The public displayed PQ was 0.40. The stored validation receipt is evidence, not an input to inference.

In [ ]:
from validate import check
assert check(OUTPUT / "submission.csv") == 0
reference = json.loads(Path("validation.json").read_text())
same = receipt["sha256"] == reference["receipt"]["sha256"]
print({"matches_recorded_csv": same, "device": receipt["device"]})
if not same:
    print("Investigate backend and version differences before claiming exact replay.")

## 5. Optional descriptive evaluation on official training images
The supplied pretrained weights have seen all training observations. These diagnostics are **not held-out validation**. The fixed subset is selected by hashes before scores are computed. Metrics include PQ, binary Dice and IoU distributions, and one-to-many/many-to-one counts at the documented diagnostic threshold 0.1. The report compares the final configuration with both single-model baselines on this same set. No test annotations are required or accepted by this workflow.

In [ ]:
RUN_DIAGNOSTICS = False
if RUN_DIAGNOSTICS:
    import evaluate
    diagnostic = json.loads(Path("diagnostics.json").read_text())
    selected = diagnostic["observations"]
    paths = [TRAIN / "train_images" / f"{s}.jpeg" for s in selected]
    assert all(p.is_file() for p in paths)
    from ultralytics import YOLO
    models = [YOLO(str(p)) for p in weights]
    predictions = {s: inference.paint_and_nms(sum([inference.predict(m, p, device=receipt["device"]) for m in models], [])) for s, p in zip(selected, paths)}
    destination = OUTPUT / "training-diagnostic.csv"
    inference.write_submission(destination, predictions)
    metrics = evaluate.evaluate(destination, TRAIN / "MAGFiLO_1.0_Annotations_kaggle2026_train.json", selected)
    print({k:v for k,v in metrics.items() if k != "per_record"})

## 6. Optional new training from scratch
This is executable training code for a **new** model. It does not claim to reconstruct the upstream author's unavailable initialization or exact training split. Default prediction loads our stored final weights and never trains.

train.py exports only the official training polygons, merges chirality classes into one filament class, and preserves annotator records. All records of an observation stay in the same split. The fixed hash rule gives 566 training and 141 validation observations. A new network uses the exact recorded architecture, seed 3407, AdamW, learning rate 0.001, weight decay 0.0001, image dimension 2048, batch size 1, and at most 100 epochs. All remaining settings are explicit in train.py or provided by the pinned Ultralytics version. Standard segmentation training uses the library's box, classification, mask and applicable distribution losses.

Training requires a separate space reservation and suitable compute. On the owner's Macs, launch substantial writers through codex-storage on the actual host. Do not enable this cell just to reproduce the submission.

In [ ]:
RUN_NEW_TRAINING = False
if RUN_NEW_TRAINING:
    import train
    data = train.prepare(TRAIN, OUTPUT / "new-training")
    trained = train.train_from_scratch(data, architecture_index=0, device="0", epochs=100)

## Provenance and scientific limits
The model cards identify 707 sanctioned training observations. Full original training arguments are in provenance.json. Original initialization and dataset configuration are not independently recovered. Copies under our account improve availability; they do not establish original authorship or remove this limitation. The author must review and understand the final method. No claim of eligibility is made solely from a score or a successful replay.